## Preprocessing of the image

In [ ]:
import cv2
import os
import numpy as np

def preprocess_images_from_folder(input_folder, output_folder=None, show_preview=False):
    """
    Reads images from a folder, applies noise removal (Gaussian Blur), binarization, and skew correction.

    Args:
        input_folder (str): Path to the folder containing input images.
        output_folder (str): Optional path to save preprocessed images. If None, images are not saved.
        show_preview (bool): If True, shows before/after preview using OpenCV.

    Returns:
        processed_images (dict): Dictionary {filename: preprocessed_image_array}
    """
    def remove_noise(image):
        """Apply Gaussian Blur for noise reduction."""
        return cv2.GaussianBlur(image, (5, 5), 0)  # (5,5) is the kernel size; increase for more smoothing.

    def binarize_image(image):
        """Convert image to binary (black and white) using Adaptive Thresholding."""
        return cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 11, 2)

    def deskew(image):
        """Correct skew in text using Hough Transform."""
        coords = np.column_stack(np.where(image > 0))
        if coords.shape[0] < 10:
            return image  # Not enough text to calculate angle
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        (h, w) = image.shape
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        return cv2.warpAffine(image, M, (w, h),
                              flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    # Create output folder if needed
    if output_folder:
        os.makedirs(output_folder, exist_ok=True)

    processed_images = {}

    for filename in os.listdir(input_folder):
        if filename.lower().endswith((".png", ".jpg", ".jpeg", ".tiff", ".bmp")):
            filepath = os.path.join(input_folder, filename)
            image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)

            if image is None:
                print(f" Unable to read {filename}. Skipping.")
                continue

            denoised = remove_noise(image)  # Apply Gaussian Blur
            binarized = binarize_image(denoised)  # Adaptive Thresholding
            deskewed = deskew(binarized)  # Deskew

            processed_images[filename] = deskewed

            if output_folder:
                out_path = os.path.join(output_folder, filename)
                cv2.imwrite(out_path, deskewed)

            if show_preview:
                cv2.imshow("Original", image)
                cv2.imshow("Preprocessed", deskewed)
                cv2.waitKey(0)
                cv2.destroyAllWindows()

    return processed_images

In [ ]:
input_folder = "/Users/krishnanand/Desktop/CVIP project"
output_folder = "/Users/krishnanand/Desktop/CVIP project"

processed = preprocess_images_from_folder(input_folder, output_folder, show_preview=True)

print(f"✅ Preprocessed {len(processed)} images.")


2025-03-21 19:07:02.367 python[3158:90395] +[IMKClient subclass]: chose IMKClient_Modern
2025-03-21 19:07:02.367 python[3158:90395] +[IMKInputSession subclass]: chose IMKInputSession_Modern


✅ Preprocessed 1 images.


: 

## Language Detection

In [1]:
import fasttext
import torch

# Detect the best available device
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Ensure correct path to model file
model_path = "/Users/krishnanand/Documents/Git/Dependencies/lid.176.bin"  # Update this path if necessary
model = fasttext.load_model(model_path)

def detect_language(text):
    """Detect language of given text using FastText."""
    predictions = model.predict(text)
    lang_code = predictions[0][0].replace('__label__', '')
    return lang_code

text = "Bonjour, comment allez-vous?"
detected_lang = detect_language(text)
print(f"Detected Language: {detected_lang}")

Using device: mps
Detected Language: fr


## Summarization  

In [2]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else (0 if torch.backends.mps.is_available() else -1)

# Load summarization pipeline (T5-Base)
summarizer = pipeline("summarization", model="t5-base", device=device)

# Example input text
text = """
Artificial intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems. 
These processes include learning, reasoning, and self-correction. AI is one of the most exciting and disruptive technologies 
of our time, influencing everything from healthcare to transportation.
"""

# Generate summary
summary = summarizer(text, max_length=100, min_length=30, do_sample=False)
summary_text = summary[0]['summary_text']
print("Summary in English:", summary_text)

# Load translation pipeline with a better model
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es", device=device)

# Translate the summary
translated_summary = translator(summary_text)
print("Translated Summary in Spanish:", translated_summary[0]['translation_text'])

Device set to use mps:0
Your max_length is set to 100, but your input_length is only 58. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=29)


Summary in English: artificial intelligence (AI) is the simulation of human intelligence processes . it includes learning, reasoning, and self-correction . AI is one of the most exciting and disruptive technologies of our time .


/opt/anaconda3/lib/python3.9/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use mps:0


Translated Summary in Spanish: inteligencia artificial (AI) es la simulación de los procesos de inteligencia humana . incluye el aprendizaje, el razonamiento y la auto-corrección . AI es una de las tecnologías más emocionantes y perturbadoras de nuestro tiempo .


## Translator

In [3]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else (0 if torch.backends.mps.is_available() else -1)

# Load the NLLB model for French to Hindi translation
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M", device=device)

# Example French text
text = "Bonjour, comment allez-vous aujourd'hui?"

# Translate the text from French to English
translated_text = translator(text, src_lang="fra_Latn", tgt_lang="eng_Latn")
english_translation = translated_text[0]['translation_text']
print("Translated to English:", english_translation)

# Now, translate from English to Hindi
translated_to_hindi = translator(english_translation, src_lang="eng_Latn", tgt_lang="hin_Deva")
print("Translated to Hindi:", translated_to_hindi[0]['translation_text'])

Device set to use mps:0


Translated to English: Hi, how are you today?
Translated to Hindi: हाय, आज आप कैसे हैं?


## TTS

In [4]:
from TTS.api import TTS
import torch

# Detect best available device (CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Load a single-language English TTS model and move it to the selected device
tts = TTS("tts_models/en/ljspeech/tacotron2-DDC").to(device)

# Text to convert
text = ("The error 'Model is not multi-lingual but language is provided' means that "
        "the model does not support multiple languages. This model can only generate speech in English, "
        "so specifying language (French) is invalid.")

# Convert text to speech and save as an audio file
tts.tts_to_file(text=text, file_path="/Users/krishnanand/Documents/Git/Tests/test123.wav")

print("Speech synthesis complete! Output saved as 'test123.wav'.")

Using device: mps
 > tts_models/en/ljspeech/tacotron2-DDC is already downloaded.
 > vocoder_models/en/ljspeech/hifigan_v2 is already downloaded.
 > Using model: Tacotron2
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:2.718281828459045
 | > hop_length:256
 | > win_length:1024
 > Model's reduction rate `r` is set to: 1
 > Vocoder Model: hifigan
 > 

/opt/anaconda3/lib/python3.9/site-packages/TTS/utils/io.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_location=map_location, **kwargs)


Removing weight norm...
 > Text splitted to sentences.
["The error 'Model is not multi-lingual but language is provided' means that the model does not support multiple languages.", 'This model can only generate speech in English, so specifying language (French) is invalid.']
 > Processing time: 4.433551073074341
 > Real-time factor: 0.2702338599106845
Speech synthesis complete! Output saved as 'test123.wav'.


In [3]:
!pip install colabcode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 740.6 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 17.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.8/82.8 kB 12.0 MB/s eta 0:00:00
  Using cached jupyter_client-8.6.3-py3-none-any.whl (106 kB)
  Using cached tornado-6.4.2-cp38-abi3-macosx_10_9_universal2.whl (436 kB)
  Using cached traitlets-5.14.3-py3-none-any.whl (85 kB)
  Using cached jupyter_core-5.7.2-py3-none-any.whl (28 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.8/529.8 kB 21.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.8/529.8 kB 23.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 22.9 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.8/529.8 kB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.8/529.8 kB 20.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 23.1 MB/s eta 0: